# Question

Find whether each user's **2nd sale** (as a seller) has an item brand that matches their **favorite brand**. Return `user_id`, `favorite_brand`, `order_date`, `item_brand`, `rnk`, and a `flag` column (`True`/`False`).

**Tables:** `b_sql.b_practice.users`, `b_sql.b_practice.orders`, `b_sql.b_practice.items`

In [0]:
%sql
-- ============================================================
-- Cell 2: Create and populate `users`, `orders`, and `items` tables
-- ============================================================
create table b_sql.b_practice.users (
user_id         int     ,
 join_date       date    ,
 favorite_brand  varchar(50));

 create table b_sql.b_practice.orders (
 order_id       int     ,
 order_date     date    ,
 item_id        int     ,
 buyer_id       int     ,
 seller_id      int 
 );

 create table b_sql.b_practice.items
 (
 item_id        int     ,
 item_brand     varchar(50)
 );
  insert into b_sql.b_practice.users values (1,'2019-01-01','Lenovo'),(2,'2019-02-09','Samsung'),(3,'2019-01-19','LG'),(4,'2019-05-21','HP');

 insert into b_sql.b_practice.items values (1,'Samsung'),(2,'Lenovo'),(3,'LG'),(4,'HP');

 insert into b_sql.b_practice.orders values (1,'2019-08-01',4,1,2),(2,'2019-08-02',2,1,3),(3,'2019-08-03',3,2,3),(4,'2019-08-04',1,4,2)
 ,(5,'2019-08-04',1,3,4),(6,'2019-08-05',2,2,4);


In [0]:
%sql
-- ============================================================
-- Cell 3: Preview the orders table
-- ============================================================
select * from b_sql.b_practice.orders

In [0]:
%sql
-- ============================================================
-- Cell 4: Preview the items table
-- ============================================================
select * from b_sql.b_practice.items

In [0]:
%sql
-- ============================================================
-- Cell 5: Preview the users table
-- ============================================================
select * from b_sql.b_practice.users

In [0]:
%sql
-- ============================================================
-- Cell 6: SQL solution — check if each user's 2nd sale brand
-- matches their favorite brand
-- CTE 1: Rank each seller's orders by order_date
-- CTE 2: Join users with their 2nd-ranked order + items for brand
-- Final: Flag True/False if item_brand = favorite_brand
-- ============================================================
with sell_rnk as (select *, row_number() over(partition by seller_id order by order_date  ) as rnk  from  b_sql.b_practice.orders ),
order as (select user_id,join_date,favorite_brand ,order_id,order_date,item_brand from
 b_sql.b_practice.users u left join 
sell_rnk o on o.seller_id = u.user_id and rnk=2
 left join b_sql.b_practice.items i on o.item_id = i.item_id 
 )
-- Final: Flag True if the 2nd sale's item brand matches the user's favorite brand
select *,(case when item_brand = favorite_brand  then True else False end)is_favorite from order 

In [0]:
# ============================================================
# Cell 7: Load users, orders, and items into PySpark DataFrames
# ============================================================
df_users=spark.read.table("b_sql.b_practice.users")
df_orders=spark.read.table("b_sql.b_practice.orders")
df_items=spark.read.table("b_sql.b_practice.items")
# Preview all three DataFrames
df_items.display()
df_orders.display()
df_users.display()

In [0]:
# ============================================================
# Cell 8: Import PySpark SQL functions, types, and Window
# (Window is needed for row_number() in the solution)
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# ============================================================
# Cell 9: PySpark solution — check if each user's 2nd sale brand
# matches their favorite brand
# Step 1: Rank each seller's orders by order_date (find 2nd sale)
# Step 2: Join users with their 2nd-ranked order + items for brand
# Step 3: Flag True/False if item_brand = favorite_brand
# ============================================================
w=Window.partitionBy("seller_id").orderBy(col("order_date"))
df_rnk_order=(df_orders.withColumn("rnk",row_number().over(w)))

# Step 2: Join users (as sellers) with their 2nd-ranked order, then join items for the brand
df_join=df_users.alias("u").join(df_rnk_order.alias("o"), ((df_users["user_id"]==df_orders["seller_id"]) & (df_orders["rnk"]==2)),"left").join(df_items.alias("i"),col("o.item_id")==col("i.item_id"),"left")

# Step 3: Select relevant columns and flag True if the 2nd sale's item brand equals the user's favorite brand
df_rnk=df_join.select("user_id","favorite_brand","order_date","item_brand","rnk",
                     when((col("favorite_brand")==col("item_brand")),True
                         ).otherwise(False).alias("flag")).display()


In [0]:
# Scratch cell